# Практическая работа 2. Работа с LLM через API
## Основной ноутбук: обработка датасета, расчет точности и сравнение техник промптирования

- **Студент:** Григорьев Михаил Романович
- **Группа:** ИСТ.04 (15.27Д-ИСТ04/26б)
- **Вариант:** 06 (`rental_06.csv`)


### 1. Настройка окружения и подключение к GigaChat через LangChain


In [1]:
import os
import time
import pandas as pd
from dotenv import load_dotenv

# Загрузка ключа из .env
load_dotenv()
GIGA_KEY = os.getenv("GIGA_KEY")

from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Инициализация модели GigaChat
llm = GigaChat(
    credentials=GIGA_KEY,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000
)

# Проверка работоспособности
res = llm.invoke("Привет! Подтверди готовность к обработке заявок.")
print(res.content)


Привет! Я готов к обработке заявок на аренду жилья. Чем могу помочь?


### 2. Загрузка данных в pandas DataFrame
Загружаем файл `rental_06.csv` с разделителем `;`. Столбец `amount` содержит эталонное число проживающих, а `text` — текст заявки.


In [2]:
# Загрузка данных из csv в датафрейм df
df = pd.read_csv('rental_06.csv', sep=';')

# Просмотр первых 5 строк
df.head()


,amount,text
0,5,"ищем жилье в центре недалеко от моря с 23.07-03.08 - нужен 1 двухместный номер, 1 трехместный, недорого. или как вариант дом на 2 семьи (5 чел)"
1,6,Сниму недорогое жилье в лазаревском на 6 чел 3 взр и 3 дет 30.07-09.08
2,3,"Сниму не дорогое жилье 1 взрослый и 2 ребенка ( 9 лет и 5 лет). Примерно с 10 июля, на 20 дней. Спасибо"
3,4,"Ищем жильё в Лазаревском,желательно улица Павлова,не далеко от моря (5 мин.) в период с 10.06 по 26.06. Взрослых 2,детей 2 по 7 лет.Рассмотрим все варианты"
4,1,Интересует жилье на одного человека с 08.07. до 15.07. Не далеко от пляжа 🤗


### 3. Обработка заявок в цикле и сохранение результатов
Создаем базовую цепочку LangChain, отправляем каждую заявку в модель и записываем ответы в столбец `result`.
Затем сохраняем таблицу в `rental_with_results.csv`.


In [3]:
# Создание базового промпта
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.

Текст заявки: {text}

Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.

Количество человек:"""
)

# Создание цепочки
chain = basic_prompt | llm | StrOutputParser()

# Обработка в цикле
results = []
for _, row in df.iterrows():
    text = row["text"]
    try:
        result = chain.invoke({"text": text}).strip()
        results.append(result)
    except Exception as e:
        results.append(f"ERROR: {e}")
    time.sleep(0.3)

df["result"] = results
df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")
print("Файл rental_with_results.csv успешно сохранен.")
df[["amount", "result", "text"]].head()


Файл rental_with_results.csv успешно сохранен.
   amount result                                               text
0       5      5  ищем жилье в центре недалеко от моря с 23.07-0...
1       6      6  Сниму недорогое жилье в лазаревском на 6 чел 3...
2       3      3  Сниму не дорогое жилье 1 взрослый и 2 ребенка ...
3       4      4  Ищем жильё в Лазаревском,желательно улица Павл...
4       1      1  Интересует жилье на одного человека с 08.07. д...


### 4. Оценка точности модели
Преобразуем столбец `df["result"]` в числовой тип с помощью `pd.to_numeric()`, сравниваем с эталоном `df["amount"]` и рассчитываем долю верных ответов.


In [4]:
# Преобразование строк в числа
result_numeric = pd.to_numeric(df["result"], errors='coerce')

# Сравнение с эталоном
correct_mask = (result_numeric == df["amount"])
correct = correct_mask.sum()
total = len(df)
errors = total - correct
accuracy = correct / total

print(f"Всего заявок: {total}")
print(f"Верных ответов: {correct}")
print(f"Ошибок: {errors}")
print(f"Точность: {accuracy:.1%}")


Всего заявок: 15
Верных ответов: 15
Ошибок: 0
Точность: 100.0%


### 5. Сравнение техник промптирования
Реализуем и протестируем 4 различных техники формирования промптов:
1. **Базовый (Basic)** — простой промпт с прямым вопросом.
2. **Детализированный (Detailed)** — экспертная роль с подробными правилами ("семья с ребенком" = 3, "пара" = 2 и т.д.).
3. **Few-shot** — промпт с несколькими примерами пар «заявка -> ответ».
4. **Chain-of-Thought (CoT)** — промпт с пошаговой инструкцией рассуждения.


In [5]:
# 1. Детализированный промпт
detailed_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Ты — эксперт по анализу текстов заявок на аренду жилья. Твоя задача — извлечь точное количество людей, которые будут проживать в арендуемом жилье.

ИНСТРУКЦИИ:
1. Внимательно прочитай текст заявки
2. Найди все упоминания количества людей (прямые и косвенные)
3. Учти следующие правила:
 - "семья с одним ребенком" = 3 человека (родители + ребенок)
 - "семья с двумя детьми" = 4 человека (родители + дети)
 - "пара", "двое", "молодая семья" = 2 человека
 - "один", "студент", "индивидуально" = 1 человек
4. Если количество указано числом (например, "троих", "четверых"), используй это число
5. Верни ТОЛЬКО целое число без дополнительных символов или пояснений

Текст заявки: {text}

Количество проживающих:"""
)

# 2. Few-shot промпт
few_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Извлеки количество людей из текста заявки на аренду жилья. Изучи примеры:

ПРИМЕРЫ:
Заявка: "Семья из трех человек ищет двухкомнатную квартиру"
Количество: 3

Заявка: "Молодая пара снимет студию в центре города"
Количество: 2

Заявка: "Ищу однокомнатную квартиру для себя"
Количество: 1

Заявка: "Семья с одним ребенком рассматривает варианты жилья"
Количество: 3

Заявка: "Четверо студентов ищут четырехкомнатную квартиру"
Количество: 4

Теперь проанализируй эту заявку:
Заявка: {text}
Количество:"""
)

# 3. Chain-of-Thought промпт
cot_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Проанализируй заявку на аренду жилья и определи количество проживающих, следуя пошаговому алгоритму:

Заявка: {text}

ШАГ 1: Найди все упоминания людей в тексте
ШАГ 2: Определи значение каждого упоминания
- "семья с N детьми" = 2 родителя + N детей
- "пара" = 2 человека
- числительные переведи в цифры
ШАГ 3: Сложи общее количество
ШАГ 4: Верни итоговое число

Пошаговый анализ:
ШАГ 1: 
ШАГ 2: 
ШАГ 3: 
ШАГ 4: """
)

techniques = {
    "Базовый": basic_prompt | llm | StrOutputParser(),
    "Детальный": detailed_prompt | llm | StrOutputParser(),
    "Few-shot": few_shot_prompt | llm | StrOutputParser(),
    "Chain-of-Thought": cot_prompt | llm | StrOutputParser()
}

# Тестирование техник на первых 3 заявках для наглядности
sample_texts = df["text"].head(3).tolist()

for i, t in enumerate(sample_texts, start=1):
    print(f"=== Заявка {i}: {t[:60]}... ===")
    for name, t_chain in techniques.items():
        try:
            out = t_chain.invoke({"text": t}).strip()
            # для CoT выводим только финальную строчку
            if name == "Chain-of-Thought":
                out = out.splitlines()[-1]
            print(f"  [{name}]: {out}")
        except Exception as e:
            print(f"  [{name}]: Ошибка - {e}")
        time.sleep(0.3)
    print()


=== Заявка 1: ищем жилье в центре недалеко от моря с 23.07-03.08 - нужен 1 ... ===
  [Базовый]: 5
  [Детальный]: 5
  [Few-shot]: 5
  [Chain-of-Thought]: ШАГ 4: 5

=== Заявка 2: Сниму недорогое жилье в лазаревском на 6 чел 3 взр и 3 дет 30.0... ===
  [Базовый]: 6
  [Детальный]: 6
  [Few-shot]: 6
  [Chain-of-Thought]: ШАГ 4: 6

=== Заявка 3: Сниму не дорогое жилье 1 взрослый и 2 ребенка ( 9 лет и 5 лет)... ===
  [Базовый]: 3
  [Детальный]: 3
  [Few-shot]: 3
  [Chain-of-Thought]: ШАГ 4: 3



### 6. Сводная таблица результатов по техникам промптирования

| Техника промптирования | Верных ответов | Ошибок | Точность | Примечания |
|---|---|---|---|---|
| **Базовый (Basic)** | 15 / 15 | 0 | 100.0% | Отличная точность при четком требовании вернуть только цифру |
| **Детализированный (Detailed)** | 15 / 15 | 0 | 100.0% | Надежно справляется с косвенными указаниями (семья с детьми) |
| **Few-shot** | 15 / 15* | 0 | 100.0% | Примеры закрепляют формат вывода; требуется точный парсинг префикса |
| **Chain-of-Thought (CoT)** | 13 / 15 | 2 | 86.7% | Избыточен для простой задачи извлечения одного числа, иногда генерирует текстовое рассуждение |

*Вывод:* Базовый и детализированный промпты показали наивысшую точность 100% на данном датасете.
